In [3]:
import pandas as pd
import requests
import folium

def load_noaa_events(csv_path='noaa_21-25_with_huc_08.csv'):
    """Loads NOAA dataset and parses datetime columns."""
    df = pd.read_csv(csv_path)
    df['BEGIN_DT'] = pd.to_datetime(df['BEGIN_DATE_TIME'])
    df['END_DT'] = pd.to_datetime(df['END_DATE_TIME'])
    return df

def generate_ifc_interactive_map(episode_id, episode_rows, hwms_df=None):
    """
    Creates an interactive Folium Leaflet map displaying:
    1. The NOAA Episode Bounding Box
    2. USGS High-Water Marks (if passed in)
    3. The Iowa Flood Center (IFC) Inundation WMS Layer
    """
    # Calculate center point for map view
    min_lat, max_lat = episode_rows['BEGIN_LAT'].min(), episode_rows['END_LAT'].max()
    min_lon, max_lon = episode_rows['BEGIN_LON'].min(), episode_rows['END_LON'].max()
    
    center_lat = (min_lat + max_lat) / 2
    center_lon = (min_lon + max_lon) / 2
    
    # Base Folium Map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="CartoDB positron")
    
    # 1. NOAA Episode Extent Bounding Box
    folium.Rectangle(
        bounds=[[min_lat, min_lon], [max_lat, max_lon]],
        color="#1f78b4",
        weight=2.5,
        fill=True,
        fill_color="#1f78b4",
        fill_opacity=0.08,
        popup=f"NOAA Episode Bounds: {episode_id}"
    ).add_to(m)
    
    # 2. Iowa Flood Center (IFC) WMS Inundation Overlay
    ifc_wms_url = "https://ifis.iowafloodcenter.org/arcgis/services/IFIS/InundationMaps/MapServer/WMSServer"
    folium.WmsTileLayer(
        url=ifc_wms_url,
        layers="0",
        name="Iowa Flood Center (IFC) Inundation",
        fmt="image/png",
        transparent=True,
        overlay=True,
        control=True,
        attr="Iowa Flood Center / IIHR - Hydroscience & Engineering"
    ).add_to(m)
    
    # 3. USGS High-Water Marks Markers (if provided)
    if hwms_df is not None and not hwms_df.empty:
        hwm_group = folium.FeatureGroup(name="USGS High-Water Marks")
        lat_col = 'latitude_dd' if 'latitude_dd' in hwms_df.columns else 'latitude'
        lon_col = 'longitude_dd' if 'longitude_dd' in hwms_df.columns else 'longitude'
        
        for idx, row in hwms_df.iterrows():
            elev = row.get('elevFt', 'N/A')
            hwm_type = row.get('hwmTypeName', 'HWM')
            popup_txt = f"<b>{hwm_type}</b><br>Elev: {elev} ft<br>ID: {row.get('hwmID', 'N/A')}"
            
            folium.CircleMarker(
                location=[row[lat_col], row[lon_col]],
                radius=5,
                color="#e31a1c",
                fill=True,
                fill_color="#fb9a99",
                fill_opacity=0.85,
                popup=popup_txt
            ).add_to(hwm_group)
            
        hwm_group.add_to(m)

    # Add Layer Control to toggle overlays on/off
    folium.LayerControl().add_to(m)
    
    return m

def run_pipeline():
    df = load_noaa_events()
    
    user_input = input("Enter NEW_EPISODE_ID to generate map (default '191899_0'): ").strip()
    if not user_input:
        user_input = "191899_0"
        
    episode_rows = df[df['NEW_EPISODE_ID'] == user_input]
    if episode_rows.empty:
        print(f"NEW_EPISODE_ID '{user_input}' not found.")
        return
        
    print(f"\nBuilding IFC Map for Episode {user_input}...")
    
    # Generate Map
    m = generate_ifc_interactive_map(user_input, episode_rows)
    
    # Save to local HTML file
    filename = f"ifc_map_episode_{user_input.replace('/', '_')}.html"
    m.save(filename)
    print(f"✓ Saved interactive map to '{filename}'!")
    
    return m

if __name__ == "__main__":
    # In Jupyter, assigning to 'm' and calling 'm' at the end renders the map inline!
    m = run_pipeline()


Building IFC Map for Episode 191899_0...
✓ Saved interactive map to 'ifc_map_episode_191899_0.html'!


In [5]:
import requests
import json
import pandas as pd
import folium

def get_ifis_inundation_data(site_id=598):
    """
    Queries the IFIS backend PHP script for flood map parameters for a given ID.
    """
    url = f"https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id={site_id}"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://ifis.iowafloodcenter.org/ifis/app/?snap_view=fmap"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # Parse response
        data = response.json()
        print(f"✓ Successfully fetched IFIS inundation metadata for ID {site_id}!")
        return data
        
    except Exception as e:
        print(f"Error fetching data from IFIS endpoint: {e}")
        return None

# --- Quick Test ---
ifis_meta = get_ifis_inundation_data(598)

if ifis_meta:
    print("\nEndpoint Response Payload:")
    print(json.dumps(ifis_meta, indent=2))

Error fetching data from IFIS endpoint: Expecting value: line 1 column 3 (char 2)


In [7]:
import requests
import json

# 1. Fetch metadata for Iowa City (id=598)
import requests
import ast

url = "https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id=598"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Referer": "https://ifis.iowafloodcenter.org/ifis/app/?snap_view=fmap"
}

response = requests.get(url, headers=headers)

# Parse response safely using ast.literal_eval instead of response.json()
data = ast.literal_eval(response.text)

# Extract stage-based KMZ links
stage_maps = data[2][0][0]  # Array of [url, stage_ft, flow_cfs, ...]

print("Successfully parsed IFIS payload!\n")
for map_entry in stage_maps[:5]:
    kmz_url, stage, flow = map_entry[0], map_entry[1], map_entry[2]
    print(f"Stage {stage} ft ({flow} cfs) -> {kmz_url}")

# 3. Download a specific KMZ (e.g., 100-year flood layer)
kmz_100yr_url = "https://iowawis.org/layers/inundation/iowa_city/ff100.0.kmz"
r = requests.get(kmz_100yr_url)

with open("iowa_city_100yr_flood.kmz", "wb") as f:
    f.write(r.content)
print("\n✓ Downloaded 'iowa_city_100yr_flood.kmz'!")

Successfully parsed IFIS payload!

Stage 17 ft (7180 cfs) -> https://iowawis.org/layers/inundation/iowa_city/017.0.kmz
Stage 17.5 ft (7780 cfs) -> https://iowawis.org/layers/inundation/iowa_city/017.5.kmz
Stage 18 ft (8400 cfs) -> https://iowawis.org/layers/inundation/iowa_city/018.0.kmz
Stage 18.5 ft (9030 cfs) -> https://iowawis.org/layers/inundation/iowa_city/018.5.kmz
Stage 19 ft (9600 cfs) -> https://iowawis.org/layers/inundation/iowa_city/019.0.kmz

✓ Downloaded 'iowa_city_100yr_flood.kmz'!


In [11]:
import ast
import io
import os
import zipfile
import pandas as pd
import requests

# Map NOAA location names / counties to IFIS Community IDs
COMMUNITY_IFIS_MAP = {
    'IOWA CITY': 598,
    'JOHNSON': 598,
    'CEDAR RAPIDS': 501,
    'LINN': 501,
    'CEDAR FALLS': 503,
    'BLACK HAWK': 503,
    'DES MOINES': 502,
    'POLK': 502,
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://ifis.iowafloodcenter.org/ifis/app/?snap_view=fmap',
}


def get_ifis_kmz_urls(ifis_id):
  """Queries IFIS backend endpoint and extracts all available KMZ download links."""
  url = f'https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id={ifis_id}'
  try:
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()

    # Safely parse single-quoted JS literal array
    data = ast.literal_eval(res.text)
    kmz_links = []

    # Discrete stage maps
    if len(data) > 2 and data[2] and data[2][0]:
      for map_item in data[2][0][0]:
        kmz_links.append(map_item[0])

    # Recurrence interval maps (10yr, 100yr, 500yr)
    if len(data) > 2 and len(data[2]) > 1 and data[2][1]:
      for map_item in data[2][1][0]:
        kmz_links.append(map_item[0])

    return kmz_links
  except Exception as e:
    print(f'  [-] Failed to fetch metadata for IFIS ID {ifis_id}: {e}')
    return []


def download_maps_for_new_episode_id(csv_path):
  # 1. Load NOAA CSV
  df = pd.read_csv(csv_path, low_memory=False)

  # Standardize headers to uppercase, but preserve exact NEW_EPISODE_ID column name
  col_map = {col: col.upper().strip() for col in df.columns}
  df = df.rename(columns=col_map)

  # Check if NEW_EPISODE_ID exists
  if 'NEW_EPISODE_ID' not in df.columns:
    print("[-] 'NEW_EPISODE_ID' column not found in your CSV.")
    print(f'    Available columns: {df.columns.tolist()}')
    return

  # 2. Ask user for NEW_EPISODE_ID
  user_ep = input(
      '\nEnter the NEW_EPISODE_ID to query (e.g., 191899_0): '
  ).strip()

  if not user_ep:
    print('[-] No NEW_EPISODE_ID provided. Exiting.')
    return

  # Filter CSV for that NEW_EPISODE_ID (as string to prevent type mismatch)
  ep_rows = df[df['NEW_EPISODE_ID'].astype(str) == str(user_ep)]

  if ep_rows.empty:
    print(f"[-] NEW_EPISODE_ID '{user_ep}' was not found in your CSV.")
    return

  print(
      f"\n[+] Found {len(ep_rows)} record(s) for NEW_EPISODE_ID '{user_ep}'."
  )

  # 3. Find matching IFIS Community IDs based on location columns
  target_ifis_ids = set()
  for _, row in ep_rows.iterrows():
    cz_name = str(row.get('CZ_NAME', '')).upper().strip()
    narrative = str(row.get('EVENT_NARRATIVE', '')).upper()

    for key, ifis_id in COMMUNITY_IFIS_MAP.items():
      if key in cz_name or key in narrative:
        print(
            f"  [>] Matched location '{key}' -> Assigning IFIS Community ID"
            f' {ifis_id}'
        )
        target_ifis_ids.add(ifis_id)

  if not target_ifis_ids:
    print(
      '  [*] No specific community string matched. Defaulting to Iowa City'
      ' (598)...'
  )
    target_ifis_ids.add(598)

  # 4. Fetch KMZ links
  all_kmz_urls = set()
  for ifis_id in target_ifis_ids:
    urls = get_ifis_kmz_urls(ifis_id)
    all_kmz_urls.update(urls)

  if not all_kmz_urls:
    print(
        '[-] No inundation KMZ files found on IFIS for NEW_EPISODE_ID'
        f" '{user_ep}'."
    )
    return

  print(
      f"\n[+] Fetching {len(all_kmz_urls)} KMZ files for NEW_EPISODE_ID"
      f" '{user_ep}'..."
  )

  # 5. Download and package into ZIP archive
  safe_ep_str = str(user_ep).replace('/', '_')
  zip_filename = f'episode_{safe_ep_str}_inundation_maps.zip'
  zip_buffer = io.BytesIO()

  with zipfile.ZipFile(
      zip_buffer, mode='w', compression=zipfile.ZIP_DEFLATED
  ) as zf:
    for kmz_url in sorted(all_kmz_urls):
      fname = kmz_url.split('/')[-1]
      try:
        r = requests.get(kmz_url, timeout=15)
        if r.status_code == 200:
          zf.writestr(fname, r.content)
          print(f'  ✓ Added {fname}')
      except Exception as e:
        print(f'  [-] Error downloading {kmz_url}: {e}')

  # Save to disk
  with open(zip_filename, 'wb') as f:
    f.write(zip_buffer.getvalue())

  print(f"\n[✓] Done! Saved all inundation maps to '{zip_filename}'.")


# --- Run Script ---
download_maps_for_new_episode_id('noaa_21-25_with_huc_08.csv')



[+] Found 4 record(s) for NEW_EPISODE_ID '194228_0'.
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598

[+] Fetching 41 KMZ files for NEW_EPISODE_ID '194228_0'...
  ✓ Added 017.0.kmz
  ✓ Added 017.5.kmz
  ✓ Added 018.0.kmz
  ✓ Added 018.5.kmz
  ✓ Added 019.0.kmz
  ✓ Added 019.5.kmz
  ✓ Added 020.0.kmz
  ✓ Added 020.5.kmz
  ✓ Added 021.0.kmz
  ✓ Added 021.5.kmz
  ✓ Added 022.0.kmz
  ✓ Added 022.5.kmz
  ✓ Added 023.0.kmz
  ✓ Added 023.5.kmz
  ✓ Added 024.0.kmz
  ✓ Added 024.5.kmz
  ✓ Added 025.0.kmz
  ✓ Added 025.5.kmz
  ✓ Added 026.0.kmz
  ✓ Added 026.5.kmz
  ✓ Added 027.0.kmz
  ✓ Added 027.5.kmz
  ✓ Added 028.0.kmz
  ✓ Added 028.5.kmz
  ✓ Added 029.0.kmz
  ✓ Added 029.5.kmz
  ✓ Added 030.0.kmz
  ✓ Added 030.5.kmz
  ✓ Added 031.0.kmz
  ✓ Added 031.5.kmz
  ✓ Added 032.0.kmz
  ✓ Added 032.5.kmz
  ✓ Added 033.0.kmz
  ✓ Added 033.5.kmz
  

In [12]:
import ast
import io
import os
import zipfile
import pandas as pd
import requests

# Map NOAA location names / counties to IFIS Community IDs
COMMUNITY_IFIS_MAP = {
    'IOWA CITY': 598,
    'JOHNSON': 598,
    'CEDAR RAPIDS': 501,
    'LINN': 501,
    'CEDAR FALLS': 503,
    'BLACK HAWK': 503,
    'DES MOINES': 502,
    'POLK': 502,
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://ifis.iowafloodcenter.org/ifis/app/?snap_view=fmap',
}


def get_ifis_kmz_items(ifis_id):
  """Queries IFIS backend endpoint and extracts KMZ download links paired with their flood extent names.

  Returns a list of tuples: (kmz_url, extent_name)
  """
  url = f'https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id={ifis_id}'
  try:
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()

    # Parse single-quoted JS array string
    data = ast.literal_eval(res.text)
    kmz_items = []

    # 1. Discrete stage maps: item structure -> [url, stage_ft, flow_cfs, ...]
    if len(data) > 2 and data[2] and data[2][0]:
      for map_item in data[2][0][0]:
        kmz_url = map_item[0]
        stage_ft = map_item[1]
        extent_name = f'stage_{stage_ft}ft'
        kmz_items.append((kmz_url, extent_name))

    # 2. Recurrence interval maps: item structure -> [url, stage_ft, flow_cfs, return_period, ...]
    if len(data) > 2 and len(data[2]) > 1 and data[2][1]:
      for map_item in data[2][1][0]:
        kmz_url = map_item[0]
        return_yr = map_item[3]
        extent_name = f'flood_{return_yr}yr'
        kmz_items.append((kmz_url, extent_name))

    return kmz_items
  except Exception as e:
    print(f'  [-] Failed to fetch metadata for IFIS ID {ifis_id}: {e}')
    return []


def download_maps_for_new_episode_id(csv_path):
  # 1. Load NOAA CSV
  df = pd.read_csv(csv_path, low_memory=False)

  # Standardize headers to uppercase
  col_map = {col: col.upper().strip() for col in df.columns}
  df = df.rename(columns=col_map)

  # Check if NEW_EPISODE_ID exists
  if 'NEW_EPISODE_ID' not in df.columns:
    print("[-] 'NEW_EPISODE_ID' column not found in your CSV.")
    print(f'    Available columns: {df.columns.tolist()}')
    return

  # 2. Ask user for NEW_EPISODE_ID
  user_ep = input(
      '\nEnter the NEW_EPISODE_ID to query (e.g., 191899_0): '
  ).strip()

  if not user_ep:
    print('[-] No NEW_EPISODE_ID provided. Exiting.')
    return

  # Filter CSV for that NEW_EPISODE_ID
  ep_rows = df[df['NEW_EPISODE_ID'].astype(str) == str(user_ep)]

  if ep_rows.empty:
    print(f"[-] NEW_EPISODE_ID '{user_ep}' was not found in your CSV.")
    return

  print(
      f"\n[+] Found {len(ep_rows)} record(s) for NEW_EPISODE_ID '{user_ep}'."
  )

  # 3. Find matching IFIS Community IDs based on location columns
  target_ifis_ids = set()
  for _, row in ep_rows.iterrows():
    cz_name = str(row.get('CZ_NAME', '')).upper().strip()
    narrative = str(row.get('EVENT_NARRATIVE', '')).upper()

    for key, ifis_id in COMMUNITY_IFIS_MAP.items():
      if key in cz_name or key in narrative:
        print(
            f"  [>] Matched location '{key}' -> Assigning IFIS Community ID"
            f' {ifis_id}'
        )
        target_ifis_ids.add(ifis_id)

  if not target_ifis_ids:
    print(
        '  [*] No specific community string matched. Defaulting to Iowa City'
        ' (598)...'
    )
    target_ifis_ids.add(598)

  # 4. Fetch KMZ links and extent names
  # Structure: { (kmz_url, extent_name), ... }
  all_kmz_items = set()
  for ifis_id in target_ifis_ids:
    items = get_ifis_kmz_items(ifis_id)
    all_kmz_items.update(items)

  if not all_kmz_items:
    print(
        '[-] No inundation KMZ files found on IFIS for NEW_EPISODE_ID'
        f" '{user_ep}'."
    )
    return

  print(
      f"\n[+] Fetching {len(all_kmz_items)} KMZ files for NEW_EPISODE_ID"
      f" '{user_ep}'..."
  )

  # 5. Download and package into ZIP archive with extent names attached
  safe_ep_str = str(user_ep).replace('/', '_')
  zip_filename = f'episode_{safe_ep_str}_inundation_maps.zip'
  zip_buffer = io.BytesIO()

  with zipfile.ZipFile(
      zip_buffer, mode='w', compression=zipfile.ZIP_DEFLATED
  ) as zf:
    for kmz_url, extent_name in sorted(all_kmz_items, key=lambda x: x[1]):
      raw_filename = kmz_url.split('/')[-1]

      # Prepend the extent name to the filename (e.g., stage_18.5ft_018.5.kmz)
      descriptive_filename = f'{extent_name}_{raw_filename}'

      try:
        r = requests.get(kmz_url, timeout=15)
        if r.status_code == 200:
          zf.writestr(descriptive_filename, r.content)
          print(f'  ✓ Added {descriptive_filename}')
      except Exception as e:
        print(f'  [-] Error downloading {kmz_url}: {e}')

  # Save ZIP to disk
  with open(zip_filename, 'wb') as f:
    f.write(zip_buffer.getvalue())

  print(f"\n[✓] Done! Saved all inundation maps to '{zip_filename}'.")


# --- Run Script ---
download_maps_for_new_episode_id('noaa_21-25_with_huc_08.csv')


[+] Found 4 record(s) for NEW_EPISODE_ID '194228_0'.
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598

[+] Fetching 41 KMZ files for NEW_EPISODE_ID '194228_0'...
  ✓ Added flood_100yr_ff100.0.kmz
  ✓ Added flood_10yr_ff010.0.kmz
  ✓ Added flood_200yr_ff200.0.kmz
  ✓ Added flood_25yr_ff025.0.kmz
  ✓ Added flood_500yr_ff500.0.kmz
  ✓ Added flood_50yr_ff050.0.kmz
  ✓ Added stage_17.5ft_017.5.kmz
  ✓ Added stage_17ft_017.0.kmz
  ✓ Added stage_18.5ft_018.5.kmz
  ✓ Added stage_18ft_018.0.kmz
  ✓ Added stage_19.5ft_019.5.kmz
  ✓ Added stage_19ft_019.0.kmz
  ✓ Added stage_20.5ft_020.5.kmz
  ✓ Added stage_20ft_020.0.kmz
  ✓ Added stage_21.5ft_021.5.kmz
  ✓ Added stage_21ft_021.0.kmz
  ✓ Added stage_22.5ft_022.5.kmz
  ✓ Added stage_22ft_022.0.kmz
  ✓ Added stage_23.5ft_023.5.kmz
  ✓ Added stage_23ft_023.0.kmz
  ✓ Added stage_24.5ft_024.5.kmz


In [14]:
!pip install fiona
import ast
import io
import os
import zipfile
import pandas as pd
import requests
import geopandas as gpd
import fiona

# Enable KML driver support in fiona/GDAL
fiona.drvsupport.supported_drivers['KML'] = 'rw'
fiona.drvsupport.supported_drivers['LIBKML'] = 'rw'

# Map NOAA location names / counties to IFIS Community IDs
COMMUNITY_IFIS_MAP = {
    'IOWA CITY': 598,
    'JOHNSON': 598,
    'CEDAR RAPIDS': 501,
    'LINN': 501,
    'CEDAR FALLS': 503,
    'BLACK HAWK': 503,
    'DES MOINES': 502,
    'POLK': 502,
}

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)',
    'Referer': 'https://ifis.iowafloodcenter.org/ifis/app/?snap_view=fmap',
}


def get_ifis_kmz_items(ifis_id):
  """Queries IFIS backend endpoint and extracts KMZ download links paired with their flood extent names.

  Returns a list of tuples: (kmz_url, extent_name)
  """
  url = f'https://ifis.iowafloodcenter.org/ifis/app/inc/inc_get_inundata.php?id={ifis_id}'
  try:
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()

    data = ast.literal_eval(res.text)
    kmz_items = []

    # 1. Discrete stage maps: item structure -> [url, stage_ft, flow_cfs, ...]
    if len(data) > 2 and data[2] and data[2][0]:
      for map_item in data[2][0][0]:
        kmz_url = map_item[0]
        stage_ft = map_item[1]
        extent_name = f'stage_{stage_ft}ft'
        kmz_items.append((kmz_url, extent_name))

    # 2. Recurrence interval maps: item structure -> [url, stage_ft, flow_cfs, return_period, ...]
    if len(data) > 2 and len(data[2]) > 1 and data[2][1]:
      for map_item in data[2][1][0]:
        kmz_url = map_item[0]
        return_yr = map_item[3]
        extent_name = f'flood_{return_yr}yr'
        kmz_items.append((kmz_url, extent_name))

    return kmz_items
  except Exception as e:
    print(f'  [-] Failed to fetch metadata for IFIS ID {ifis_id}: {e}')
    return []


def convert_kmz_bytes_to_geodataframe(kmz_bytes, extent_name):
  """Extracts internal doc.kml from raw KMZ bytes, reads it into GeoPandas,

  attaches metadata, and reprojects to NAD83 / Iowa South (EPSG:3418).
  """
  try:
    with zipfile.ZipFile(io.BytesIO(kmz_bytes)) as z:
      kml_filenames = [f for f in z.namelist() if f.endswith('.kml')]
      if not kml_filenames:
        return None

      # Read raw KML content
      kml_data = z.read(kml_filenames[0])

    # Parse KML into GeoDataFrame
    gdf = gpd.read_file(io.BytesIO(kml_data), driver='KML')

    # Assign metadata columns
    gdf['extent_name'] = extent_name

    # Reproject from WGS84 (EPSG:4326) to Iowa South (EPSG:3418) for accurate GIS area calculations
    if gdf.crs is None or gdf.crs.to_epsg() != 3418:
      gdf = gdf.to_crs(epsg=3418)

    # Calculate flooded area in acres directly in EPSG:3418
    gdf['area_acres'] = gdf.geometry.area / 4046.8564224

    return gdf
  except Exception as e:
    print(f'  [-] Could not parse KMZ into GeoDataFrame for {extent_name}: {e}')
    return None


def download_and_process_maps_for_episode(csv_path):
  # 1. Load NOAA CSV
  df = pd.read_csv(csv_path, low_memory=False)

  # Standardize headers to uppercase
  col_map = {col: col.upper().strip() for col in df.columns}
  df = df.rename(columns=col_map)

  if 'NEW_EPISODE_ID' not in df.columns:
    print("[-] 'NEW_EPISODE_ID' column not found in your CSV.")
    print(f'    Available columns: {df.columns.tolist()}')
    return

  # 2. Prompt user
  user_ep = input(
      '\nEnter the NEW_EPISODE_ID to query (e.g., 191899_0): '
  ).strip()

  if not user_ep:
    print('[-] No NEW_EPISODE_ID provided. Exiting.')
    return

  ep_rows = df[df['NEW_EPISODE_ID'].astype(str) == str(user_ep)]

  if ep_rows.empty:
    print(f"[-] NEW_EPISODE_ID '{user_ep}' was not found in your CSV.")
    return

  print(
      f"\n[+] Found {len(ep_rows)} record(s) for NEW_EPISODE_ID '{user_ep}'."
  )

  # 3. Match locations
  target_ifis_ids = set()
  for _, row in ep_rows.iterrows():
    cz_name = str(row.get('CZ_NAME', '')).upper().strip()
    narrative = str(row.get('EVENT_NARRATIVE', '')).upper()

    for key, ifis_id in COMMUNITY_IFIS_MAP.items():
      if key in cz_name or key in narrative:
        print(
            f"  [>] Matched location '{key}' -> Assigning IFIS Community ID"
            f' {ifis_id}'
        )
        target_ifis_ids.add(ifis_id)

  if not target_ifis_ids:
    print(
        '  [*] No specific community string matched. Defaulting to Iowa City'
        ' (598)...'
    )
    target_ifis_ids.add(598)

  # 4. Fetch KMZ links
  all_kmz_items = set()
  for ifis_id in target_ifis_ids:
    items = get_ifis_kmz_items(ifis_id)
    all_kmz_items.update(items)

  if not all_kmz_items:
    print(
        '[-] No inundation KMZ files found on IFIS for NEW_EPISODE_ID'
        f" '{user_ep}'."
    )
    return

  print(
      f"\n[+] Fetching {len(all_kmz_items)} KMZ files for NEW_EPISODE_ID"
      f" '{user_ep}'..."
  )

  # 5. Download, package into ZIP, and convert to GeoPackage
  safe_ep_str = str(user_ep).replace('/', '_')
  zip_filename = f'episode_{safe_ep_str}_inundation_maps.zip'
  gpkg_filename = f'episode_{safe_ep_str}_inundation_layers.gpkg'

  zip_buffer = io.BytesIO()
  gdfs = []

  with zipfile.ZipFile(
      zip_buffer, mode='w', compression=zipfile.ZIP_DEFLATED
  ) as zf:
    for kmz_url, extent_name in sorted(all_kmz_items, key=lambda x: x[1]):
      raw_filename = kmz_url.split('/')[-1]
      descriptive_filename = f'{extent_name}_{raw_filename}'

      try:
        r = requests.get(kmz_url, timeout=15)
        if r.status_code == 200:
          # Write raw KMZ to ZIP
          zf.writestr(descriptive_filename, r.content)
          print(f'  ✓ Added raw KMZ: {descriptive_filename}')

          # Parse KML into GeoDataFrame for GIS
          gdf = convert_kmz_bytes_to_geodataframe(r.content, extent_name)
          if gdf is not None and not gdf.empty:
            gdfs.append(gdf)

      except Exception as e:
        print(f'  [-] Error downloading {kmz_url}: {e}')

  # Save ZIP file
  with open(zip_filename, 'wb') as f:
    f.write(zip_buffer.getvalue())
  print(f"\n[✓] Saved raw archive to '{zip_filename}'.")

  # Save combined GeoPackage for GIS analysis
  if gdfs:
    combined_gdf = pd.concat(gdfs, ignore_index=True)

    # Clean description/style HTML columns if present in KML output
    cols_to_drop = [
        c for c in ['Description', 'StyleMap', 'styleUrl'] if c in combined_gdf
    ]
    combined_gdf = combined_gdf.drop(columns=cols_to_drop)

    combined_gdf.to_file(gpkg_filename, driver='GPKG')
    print(
        f"[✓] Created spatial GIS vector dataset: '{gpkg_filename}'"
        f' (EPSG:3418).'
    )


# --- Run Script ---
download_and_process_maps_for_episode('noaa_21-25_with_huc_08.csv')

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/24.5 MB ? eta -:--:--
   ------------------------------- -------- 19.1/24.5 MB 104.4 MB/s eta 0:00:01
   ---------------------------------------- 24.5/24.5 MB 92.8 MB/s  0:00:00

   -------------------- ------------------- 2/4 [click-plugins]
   ------------------------------ --------- 3/4 [fiona]
   ------------------------------ --------- 3/4 [fiona]
   ------------------------------ --------- 3/4 [fiona]
   ---------------------------------------- 4/4 [fiona]




[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\cfuchtman\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[+] Found 4 record(s) for NEW_EPISODE_ID '194228_0'.
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598
  [>] Matched location 'JOHNSON' -> Assigning IFIS Community ID 598

[+] Fetching 41 KMZ files for NEW_EPISODE_ID '194228_0'...
  ✓ Added raw KMZ: flood_100yr_ff100.0.kmz
  ✓ Added raw KMZ: flood_10yr_ff010.0.kmz
  ✓ Added raw KMZ: flood_200yr_ff200.0.kmz
  ✓ Added raw KMZ: flood_25yr_ff025.0.kmz
  ✓ Added raw KMZ: flood_500yr_ff500.0.kmz
  ✓ Added raw KMZ: flood_50yr_ff050.0.kmz
  ✓ Added raw KMZ: stage_17.5ft_017.5.kmz
  ✓ Added raw KMZ: stage_17ft_017.0.kmz
  ✓ Added raw KMZ: stage_18.5ft_018.5.kmz
  ✓ Added raw KMZ: stage_18ft_018.0.kmz
  ✓ Added raw KMZ: stage_19.5ft_019.5.kmz
  ✓ Added raw KMZ: stage_19ft_019.0.kmz
  ✓ Added raw KMZ: stage_20.5ft_020.5.kmz
  ✓ Added raw KMZ: stage_20ft_020.0.kmz
  ✓ Added raw KMZ: stage_21.5ft_021.5.kmz
  ✓ Added raw KMZ: stage_21ft_021.0.kmz
  ✓ Added raw KMZ